# Final Project Part 3: Analysis Notebook

## The Earth Doesn't Sit Still: 30 Days of Global Earthquakes

**Authors:** Inseo Song (inseos2), Won Roh (wonhyun2)

This notebook is where we did all the analysis and chart-building for our Part 3 article. We started from our Part 2 dashboard and added new Plotly charts that we ended up using on the final Jekyll page.

**Published article:** https://[your-username].github.io/[your-repo]/

## 1. Setup and data loading

In [1]:
!pip install plotly --quiet

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import bqplot
import ipywidgets
import traitlets

In [13]:
# Load the USGS earthquake catalog (M4.5+, past 30 days)
quakes = pd.read_csv("USGS_EarthQuake (30days).csv", parse_dates=["time"])

# Drop the timezone, bqplot interval selectors return tz-naive datetimes,
# so the index needs to be tz-naive to allow filtering later.
quakes["time"] = quakes["time"].dt.tz_localize(None)

print("Shape:", quakes.shape)
print("Date range:", quakes["time"].min(), "to", quakes["time"].max())
quakes.head()

Shape: (571, 22)
Date range: 2026-03-30 00:23:55.075000 to 2026-04-28 22:58:22.003000


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-04-28 22:58:22.003,12.3502,125.5388,35.000,4.9,mb,44,110,5.247,0.95,...,2026-04-28T23:13:32.040Z,"15 km NNE of Alugan, Philippines",earthquake,10.82,1.928,0.086,42.0,reviewed,us,us
1,2026-04-28 19:16:05.298,41.2150,83.9918,10.000,4.6,mb,59,107,3.955,0.58,...,2026-04-28T19:44:19.040Z,"104 km ESE of Kuqa, China",earthquake,6.79,1.876,0.078,49.0,reviewed,us,us
2,2026-04-28 19:13:15.642,-18.3551,-175.1875,268.705,4.5,mb,25,122,5.036,0.86,...,2026-04-28T19:30:37.040Z,"131 km WNW of Neiafu, Tonga",earthquake,14.51,15.563,0.127,18.0,reviewed,us,us
3,2026-04-28 16:32:06.338,-5.9905,146.5918,43.834,4.7,mb,116,41,3.438,0.53,...,2026-04-28T18:00:17.040Z,"86 km ENE of Kainantu, Papua New Guinea",earthquake,7.57,7.380,0.054,103.0,reviewed,us,us
4,2026-04-28 16:18:24.175,-23.1275,-175.2728,10.000,4.9,mww,44,129,6.414,0.71,...,2026-04-28T18:17:18.040Z,"200 km S of ‘Ohonua, Tonga",earthquake,11.30,1.892,0.110,8.0,reviewed,us,us


## 2. About the dataset

We're using the USGS earthquake catalog, which lists every earthquake of magnitude 4.5 or greater detected anywhere in the world over the past 30 days. There are 571 rows and 22 columns. Each row is one earthquake.

We mostly care about these five columns:

- `time`: when the earthquake happened (in UTC)
- `latitude` and `longitude`: where it happened
- `depth`: how far below the surface it started, in km
- `mag`: the magnitude (in our data this ranges from 4.5 to 7.4)
- `place`: a short text description like "15 km NNE of Alugan, Philippines"

The other 17 columns are mostly metadata about how the measurement was made, useful for double-checking the data but not something we plot directly.

In [4]:
# Quick descriptive stats on our key columns
quakes[["mag", "depth", "latitude", "longitude"]].describe()

,mag,depth,latitude,longitude
count,571.000000,571.000000,571.000000,571.000000
mean,4.841296,64.355042,3.368396,53.111566
std,0.385778,104.280966,27.736832,121.459657
min,4.500000,3.000000,-62.012300,-179.900500
25%,4.600000,10.000000,-15.360150,-63.704900
50%,4.700000,35.000000,1.157300,126.118400
75%,5.000000,58.373000,26.216200,136.081950
max,7.400000,649.858000,58.067400,179.799500


In [5]:
# Quick data quality check before we visualize
print("Missing values per key column:")
print(quakes[["time", "latitude", "longitude", "depth", "mag", "place"]].isna().sum())
print()
print(f"Magnitude range: {quakes['mag'].min():.1f} to {quakes['mag'].max():.1f}")
print(f"Depth range: {quakes['depth'].min():.1f} to {quakes['depth'].max():.1f} km")
print(f"Total events: {len(quakes)}")
print(f"Unique places: {quakes['place'].nunique()}")

Missing values per key column:
time         0
latitude     0
longitude    0
depth        0
mag          0
place        0
dtype: int64

Magnitude range: 4.5 to 7.4
Depth range: 3.0 to 649.9 km
Total events: 571
Unique places: 436


## 3. Pulling out the country/region from `place`

The `place` column is a string like "15 km NNE of Alugan, Philippines". To group earthquakes by country for our Top 10 chart, we just take whatever comes after the last comma. A few entries don't have a comma at all (like "Fiji region" or "south of the Kermadec Islands"), and for those we just keep the whole string as-is.

In [6]:
# Extract a "region" (country / area) from the place string
quakes["region"] = quakes["place"].fillna("Unknown").apply(
    lambda s: s.split(",")[-1].strip() if "," in s else s.strip()
)

quakes["region"].value_counts().head(10)

Indonesia                        160
Japan                             50
south of the Kermadec Islands     33
Russia                            25
Tonga                             21
Papua New Guinea                  18
Alaska                            17
Vanuatu                           16
Solomon Islands                   13
Philippines                       12
Name: region, dtype: int64

## 4. Part 2 dashboard (kept here for reference)

This is the bqplot dashboard from Part 2. The top chart shows how many earthquakes happened on each day, and you can drag a date range on it. The map below then updates to show only the earthquakes from the days you picked.

We kept this here so the notebook tells the full story of how we got from Part 2 to Part 3. For the actual Part 3 article, we rebuilt these in Plotly because Plotly figures can be saved as standalone HTML files and dropped into a Jekyll page with an iframe.

In [15]:
# Aggregate to daily counts for the bqplot driver plot
quakes_by_date = quakes.set_index("time").sort_index()
daily_counts = quakes_by_date.groupby(quakes_by_date.index.date).size()
daily_counts.index = pd.to_datetime(daily_counts.index)

# Driver plot: daily earthquake count over time
date_sc = bqplot.DateScale()
count_sc = bqplot.LinearScale()

date_ax = bqplot.Axis(scale=date_sc, label="Date (UTC)", tick_format="%b %d")
count_ax = bqplot.Axis(scale=count_sc, label="Number of earthquakes",
                       orientation="vertical")

bars = bqplot.Bars(x=daily_counts.index, y=daily_counts.values,
                   scales={"x": date_sc, "y": count_sc},
                   colors=["pink"])

interval_selector = bqplot.interacts.FastIntervalSelector(scale=date_sc)

fig_timeline = bqplot.Figure(
    marks=[bars], axes=[date_ax, count_ax],
    interaction=interval_selector,
    title="Daily earthquake frequency (drag to select a date range)"
)
fig_timeline

Figure(axes=[Axis(label='Date (UTC)', scale=DateScale(), tick_format='%b %d'), Axis(label='Number of earthquak…

In [8]:
# Driven plot: world map of epicenters
lon_sc = bqplot.LinearScale(min=-180, max=180)
lat_sc = bqplot.LinearScale(min=-90, max=90)
mag_sc = bqplot.ColorScale(scheme="Reds",
                           min=quakes["mag"].min(),
                           max=quakes["mag"].max())

lon_ax = bqplot.Axis(scale=lon_sc, label="Longitude (°)", grid_lines="dashed")
lat_ax = bqplot.Axis(scale=lat_sc, label="Latitude (°)",
                     orientation="vertical", grid_lines="dashed")
mag_ax = bqplot.ColorAxis(scale=mag_sc, label="Magnitude", side="right")

scatter = bqplot.Scatter(x=quakes["longitude"].values,
                         y=quakes["latitude"].values,
                         color=quakes["mag"].values,
                         scales={"x": lon_sc, "y": lat_sc, "color": mag_sc},
                         default_size=24,
                         stroke="black",
                         stroke_width=0.3)

fig_map_bqplot = bqplot.Figure(marks=[scatter], axes=[lon_ax, lat_ax, mag_ax],
                               title="Earthquake epicenters (filtered by selected date range)")

# Status label
status_label = ipywidgets.HTML(
    value=f"<b>Showing all {len(quakes)} earthquakes.</b> "
          f"Drag on the bar chart above to filter by date."
)

# Link the driver and driven plots
def on_interval_change(change):
    if change["new"] is None:
        # Brush cleared -> show everything
        scatter.x = quakes["longitude"].values
        scatter.y = quakes["latitude"].values
        scatter.color = quakes["mag"].values
        status_label.value = (f"<b>Showing all {len(quakes)} earthquakes.</b> "
                              f"Drag on the bar chart above to filter by date.")
        return
    start_date, stop_date = change["new"]
    sub = quakes_by_date.loc[start_date:stop_date]
    scatter.x = sub["longitude"].values
    scatter.y = sub["latitude"].values
    scatter.color = sub["mag"].values
    status_label.value = (f"<b>Showing {len(sub)} earthquakes</b> "
                          f"from {pd.Timestamp(start_date).date()} "
                          f"to {pd.Timestamp(stop_date).date()}.")

interval_selector.observe(on_interval_change, "selected")

# Final dashboard layout
ipywidgets.VBox([
    ipywidgets.HTML("<h3>Global Earthquake Dashboard (M4.5+, last 30 days)</h3>"),
    fig_timeline,
    status_label,
    fig_map_bqplot
])

## 5. Plotly charts for the article

Now we build the two charts that go on the Jekyll page. We picked Plotly because `fig.write_html()` gives us a self-contained HTML file we can embed directly. The hover tooltips, zoom, pan, and legend toggles all keep working on the published site, which is what we need for the "interactive" part of the rubric.

### 5.1 The main map: every earthquake from the past 30 days

This is the central interactive figure on the article. Every dot is one earthquake. The color gets darker red the stronger the quake, and the dot size gets bigger too. Readers can hover any dot for the exact time, location, depth, and magnitude.

**A few things we want to flag about how we made this plot:**

- **We used a flat lon/lat scatter instead of a real map projection.** Most public earthquake maps put the dots on a globe or a Mercator map. We considered that, but each row in our data is just a single point event, and a flat scatter lets readers click on individual quakes without the projection distorting how dense the clusters look. So this is a deliberate choice, not the most "professional-looking" option, but the one that's easiest to read.
- **Color uses a sequential `Reds` scale.** Magnitude only goes one direction (higher = stronger), so a sequential scale is the right choice. Red is also what most USGS publications use for seismic intensity, so it should feel familiar.
- **Both color and size encode magnitude.** This is technically redundant, but doubling up makes the strong earthquakes jump out without the reader having to squint at the legend.

In [16]:
fig_map_plotly = px.scatter(
    quakes,
    x="longitude",
    y="latitude",
    color="mag",
    color_continuous_scale="Reds",
    size="mag",
    size_max=18,
    hover_data={
        "place": True,
        "mag": ":.1f",
        "depth": ":.1f",
        "time": "|%Y-%m-%d %H:%M",
        "longitude": ":.2f",
        "latitude": ":.2f",
    },
    labels={
        "longitude": "Longitude (°)",
        "latitude": "Latitude (°)",
        "mag": "Magnitude",
        "depth": "Depth (km)",
        "time": "Time (UTC)",
        "place": "Location",
    },
    title="Where the Earth shook: 571 earthquakes (M4.5+) over 30 days",
)

fig_map_plotly.update_layout(
    height=600,
    xaxis=dict(range=[-180, 180], dtick=30, gridcolor="lightgray"),
    yaxis=dict(range=[-90, 90], dtick=30, gridcolor="lightgray"),
    title_font_size=20,
    font=dict(size=14),
    plot_bgcolor="#f7f9fc",
)

fig_map_plotly.show()

/home/jovyan/.local/lib/python3.8/site-packages/plotly/io/_renderers.py:58: UserWarning: Plotly version >= 6 requires JupyterLab >= 3 but you have 2.2.9 installed. To upgrade JupyterLab, please run `pip install jupyterlab --upgrade`.
  warnings.warn(


In [10]:
# Export to standalone HTML for embedding in the Jekyll site
fig_map_plotly.write_html("earthquake_map.html",
                          include_plotlyjs="cdn",
                          full_html=True)
print("Saved: earthquake_map.html")

Saved: earthquake_map.html


### 5.2 Top 10 regions

This is one of our two contextual charts. We just count earthquakes by region and show the top 10. We knew Indonesia would be high before we plotted it, but we were still kind of surprised by how big the gap is between #1 and everyone else.

**Why we made the choices we did:**

- **Horizontal bars instead of vertical.** Some of the region names are long (like "south of the Kermadec Islands") and would get squished or rotated if we used vertical bars.
- **Same `Reds` colormap as the main map.** We wanted the two charts to feel visually connected. The reader's eye should pick up that "more red = more important" without us having to explain it.
- **Bars are sorted by count, not alphabetically.** This is a Top 10 chart, so the whole point is the ranking.

In [11]:
top10 = (
    quakes["region"]
    .value_counts()
    .head(10)
    .rename_axis("Region")
    .reset_index(name="Earthquake count")
)

fig_top10 = px.bar(
    top10,
    x="Earthquake count",
    y="Region",
    orientation="h",
    color="Earthquake count",
    color_continuous_scale="Reds",
    text="Earthquake count",
    title="Top 10 regions by number of M4.5+ earthquakes (past 30 days)",
)

fig_top10.update_layout(
    height=500,
    yaxis=dict(categoryorder="total ascending", title=""),
    xaxis_title="Number of earthquakes",
    title_font_size=20,
    font=dict(size=14),
    plot_bgcolor="#f7f9fc",
    coloraxis_showscale=False,
)
fig_top10.update_traces(textposition="outside")

fig_top10.show()

/home/jovyan/.local/lib/python3.8/site-packages/plotly/io/_renderers.py:58: UserWarning: Plotly version >= 6 requires JupyterLab >= 3 but you have 2.2.9 installed. To upgrade JupyterLab, please run `pip install jupyterlab --upgrade`.
  warnings.warn(


In [12]:
# Export to standalone HTML for embedding in the Jekyll site
fig_top10.write_html("top10_regions.html",
                     include_plotlyjs="cdn",
                     full_html=True)
print("Saved: top10_regions.html")

Saved: top10_regions.html


## 5.3 Contextual visualization #2: tectonic plate boundary map

For our second contextual visualization we're using an existing image of Earth's tectonic plates instead of making our own chart. We thought about plotting the actual plate boundary dataset from `github.com/fraxen/tectonicplates` on top of our map, but the existing USGS image is much cleaner and easier for a non-expert to read at a glance, so we decided to just use that.

Since this image isn't ours, we make sure to cite it clearly on the published article (USGS, public domain, via Wikimedia Commons).

**Image we're using:**
https://upload.wikimedia.org/wikipedia/commons/thumb/8/8a/Plates_tect2_en.svg/1280px-Plates_tect2_en.svg.png

## 6. Wrap-up

A few takeaways from this dataset:

- 571 earthquakes of magnitude 4.5 or higher in 30 days, roughly 19 per day worldwide.
- Indonesia alone had 160 of them. That's about 28% of the global total, and almost three times as many as the next country (Japan, with 50).
- The geographic clustering is really clear once you put all the dots on a map. They line up almost exactly with the tectonic plate boundaries, which is why we use the plate boundary image as our second contextual visualization.

The two HTML files this notebook saves (`earthquake_map.html` and `top10_regions.html`) are what we embed as iframes on the Jekyll page.

## Data sources

- **USGS Earthquake Catalog (primary dataset):** https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php
- **Tectonic plate boundary map (contextual image):** https://commons.wikimedia.org/wiki/File:Plates_tect2_en.svg
- **Tectonic plate boundary dataset (referenced for further reading):** https://github.com/fraxen/tectonicplates
- **"Ring of Fire" background reading:** https://www.usgs.gov/programs/earthquake-hazards/science/ring-fire